# 05 · Consistencia y variabilidad

**Pregunta central 3 del Plan Maestro (segunda mitad):** ¿es estable el estilo de Jardine o cambia según el rival, la localidad (local/visitante) o la temporada?

Este bloque no mira solo el promedio agregado de cada métrica (eso ya lo vimos en los notebooks 01-03), sino la **distribución partido a partido**: qué tan dispersos están los 129 partidos alrededor de ese promedio, usando el coeficiente de variación (CV = desviación estándar / media) como medida de consistencia.

- **CV bajo** → el equipo repite un patrón similar partido tras partido (estilo consistente).
- **CV alto** → la métrica varía mucho de partido a partido (estilo inconsistente, o muy dependiente del rival/contexto).

In [ ]:
import sys
sys.path.append("../src")
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from framework_analitico import (
    normalizar_direccion_ataque,
    obtener_partidos_entrenador_equipo,
    metricas_por_partido,
    agregar_contexto_partido,
    resumen_variabilidad,
    variabilidad_por_grupo,
)

EQUIPO = "América"  # cambia esto para reaplicar el analisis a otro entrenador/equipo
ENTRENADOR = "André Soares Jardine"

events_jardine = pd.read_csv("../data/raw/events_jardine_america.csv", low_memory=False)
print(f"Eventos cargados: {len(events_jardine)}")

events_norm = normalizar_direccion_ataque(events_jardine, equipo=EQUIPO)
print(f"Partidos: {events_norm['match_id'].nunique()}")

## 1. Metadatos de partido (localidad y temporada)

La localidad (local/visitante) y la temporada NO están en el CSV de eventos — solo viven en los metadatos de partido de StatsBomb (`sb.matches`), que no se habían guardado todavía. Esta celda los descarga una sola vez y los deja cacheados en `../data/raw/partidos_jardine_america.csv` para no tener que repetir la consulta.

In [ ]:
PARTIDOS_PATH = "../data/raw/partidos_jardine_america.csv"

if os.path.exists(PARTIDOS_PATH):
    partidos = pd.read_csv(PARTIDOS_PATH)
    print(f"Metadatos de partidos cargados desde cache: {len(partidos)} partidos")
else:
    os.environ.setdefault("SB_USERNAME", "<TU_USUARIO_STATSBOMB>")
    os.environ.setdefault("SB_PASSWORD", "<TU_PASSWORD_STATSBOMB>")
    from statsbombpy import sb

    comps = sb.competitions()
    liga_mx = comps[comps.competition_name.str.contains("Liga MX", case=False, na=False)]
    liga_mx = liga_mx.drop_duplicates(subset="season_id")

    all_matches = []
    for season_id in liga_mx.season_id.tolist():
        m = sb.matches(competition_id=liga_mx.competition_id.iloc[0], season_id=season_id)
        m["season_id"] = season_id
        all_matches.append(m)
    matches_df = pd.concat(all_matches, ignore_index=True)

    mapa_temporadas = liga_mx.set_index("season_id")["season_name"].to_dict()
    matches_df["temporada"] = matches_df["season_id"].map(mapa_temporadas)

    partidos = obtener_partidos_entrenador_equipo(matches_df, entrenador=ENTRENADOR, equipo=EQUIPO)
    partidos.to_csv(PARTIDOS_PATH, index=False)
    print(f"Metadatos de partidos descargados y guardados en cache: {len(partidos)} partidos")

partidos[["match_id", "home_team", "away_team", "temporada", "match_date"]].head()

## 2. Métricas calculadas partido por partido

En vez del perfil agregado de todo el dataset (notebook 03), aquí `perfil_ofensivo_defensivo` e `indice_directividad` se corren una vez POR PARTIDO, y se les agrega la localidad, el rival y la temporada de cada uno.

In [ ]:
metricas = metricas_por_partido(events_norm, equipo=EQUIPO)
metricas = agregar_contexto_partido(metricas, partidos, equipo=EQUIPO)

print(f"Partidos con métricas calculadas: {len(metricas)}")
metricas.head()

## 3. ¿Qué tan consistente es cada métrica? (coeficiente de variación)

Métricas ordenadas de la MENOS consistente (CV más alto) a la más consistente (CV más bajo).

In [ ]:
columnas_metricas = [
    "posesion_pct", "field_tilt_pct", "field_tilt_relativo_pct", "ppda",
    "tiros_favor", "tiros_contra", "xg_favor", "xg_contra",
    "pct_pases_largos", "longitud_promedio_m",
]
columnas_metricas = [c for c in columnas_metricas if c in metricas.columns]

variabilidad = resumen_variabilidad(metricas, columnas=columnas_metricas)
variabilidad

## 4. Visualización: distribución partido a partido (no solo el promedio)

Boxplots de las métricas clave: la caja muestra dónde cae la mitad central de los partidos, y los puntos fuera de los bigotes son partidos atípicos.

In [ ]:
columnas_box = ["posesion_pct", "field_tilt_relativo_pct", "ppda", "pct_pases_largos"]
columnas_box = [c for c in columnas_box if c in metricas.columns]

fig, axes = plt.subplots(1, len(columnas_box), figsize=(4 * len(columnas_box), 5))
for ax, col in zip(axes, columnas_box):
    ax.boxplot(metricas[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor="#FFD700", edgecolor="black"),
               medianprops=dict(color="#8B0000", linewidth=2))
    ax.set_title(col)
    ax.set_xticks([])

plt.suptitle(f"Distribución partido a partido ({len(metricas)} partidos)")
plt.tight_layout()
plt.savefig("../outputs/figures/distribucion_partido_a_partido.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. ¿Cambia la consistencia según la localidad (local/visitante)?

In [ ]:
print(metricas["localidad"].value_counts())
print()

variabilidad_localidad = variabilidad_por_grupo(metricas, columna_grupo="localidad", columnas=columnas_metricas)
variabilidad_localidad.pivot(index="metrica", columns="localidad", values="media")

## 6. ¿Cambia la consistencia según la temporada?

In [ ]:
print(metricas["temporada"].value_counts())
print()

variabilidad_temporada = variabilidad_por_grupo(metricas, columna_grupo="temporada", columnas=columnas_metricas)
variabilidad_temporada.pivot(index="metrica", columns="temporada", values="media")

## 7. Síntesis: ¿qué tan consistente es el estilo de Jardine?

Se calcularon las métricas de perfil ofensivo/defensivo y directividad **partido por partido** (no solo el agregado) para los 129 partidos, y se midió su consistencia con el coeficiente de variación (CV = desviación estándar / media).

**1. Hay una división clara entre métricas de "estilo" (consistentes) y de "rendimiento" (variables):**

- Las más consistentes: **posesión** (CV 0.048, la más baja por mucho), **directividad** (% pases largos CV 0.113, longitud promedio CV 0.078) y **field tilt** (0.16-0.22). Son decisiones propias del modelo de juego, y casi no cambian de partido a partido.
- Las menos consistentes: **xG en contra** (CV 0.667), **xG a favor** (0.631), **PPDA** (0.576) y **tiros** (0.40-0.43). Dependen tanto de lo que hace América como de lo que le permite el rival, así que varían mucho más.

**2. Localidad (local/visitante): el estilo no cambia, pero la ejecución sí.** Posesión, field tilt y directividad son casi idénticos jugando en casa o de visita (diferencias de menos de 1.5pp). En cambio, en casa América presiona más intenso (PPDA 2.29 vs. 2.63 de visitante), genera más tiros (14.97 vs. 12.65) y concede menos (9.72 vs. 12.57) — el plan de juego es el mismo, pero se ejecuta mejor en casa.

**3. Temporada: hay una evolución clara, con un matiz importante.**
- **Mejora sostenida en lo defensivo:** el PPDA baja 20% entre 23/24 y 25/26 (2.79 → 2.23) y los tiros en contra caen de 12.2 a 10.1 por partido — el equipo presiona cada vez más fuerte y concede cada vez menos.
- **Mejora en el dominio territorial:** field tilt relativo sube de 48.6% a 50.8%, y el equipo se vuelve algo más directo (más pases largos, mayor longitud promedio).
- **Matiz: la calidad de las ocasiones generadas baja.** Aunque los tiros a favor suben en la última temporada (13.5 → 13.2 → 14.8), el xG a favor cae de forma sostenida (1.82 → 1.55 → 1.39), lo que implica que el xG promedio por tiro bajó de ~0.135 a ~0.094 entre la primera y la última temporada. América tira más veces pero cada tiro individual vale menos — probablemente desde posiciones más lejanas o de menor calidad. No es una historia de "todo mejora": la fase defensiva y el dominio territorial sí progresan con el tiempo, pero la generación de ocasiones de alta calidad no acompaña ese progreso.

**Lectura general para el Plan Maestro:** el estilo de Jardine (cómo construye, cuánto posee el balón) es muy estable y no depende del contexto — es una identidad definida. Lo que sí varía es el resultado de ese estilo sobre el campo (tiros, xG, PPDA), y esa variación se explica en buena parte por dos factores identificables: la localidad (mejor ejecución en casa) y la evolución temporal (mejora defensiva sostenida, con una caída paralela en la calidad de las ocasiones ofensivas que vale la pena investigar más a fondo, posiblemente en el bloque de balón parado o uso de jugadores).